# 8장 실습 ④ — 표준 데이터셋에서 DNN 대 CNN

**Keras 3 판**

§8.3에서는 **합성 도형**으로 확인했습니다. 이제 진짜 데이터로 합니다.

> **MNIST 하나만 보면 CNN이 별것 아닌 것처럼 보입니다.**
> 세 개를 나란히 놓아야 드러납니다.

## 8.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 8.1 세 데이터셋

In [ ]:
# 세 데이터셋. 뒤로 갈수록 어렵습니다.
SETS = [("MNIST", data.mnist, (28, 28, 1)),
        ("Fashion-MNIST", data.fashion_mnist, (28, 28, 1)),
        ("CIFAR-10", data.cifar10, (32, 32, 3))]
if dlbook.smoke.is_smoke():
    SETS = SETS[:1]

for name, fn, shape in SETS:
    s = fn()
    print(f"{name:<16} 학습 {len(s.x_train):>6,}  시험 {len(s.x_test):>6,}  입력 {shape}")

print()
print("★ MNIST는 숫자가 대체로 **가운데** 있습니다.")
print("  Fashion-MNIST도 옷이 가운데 놓여 있습니다.")
print("  CIFAR-10은 **사진**입니다 — 배경이 있고, 각도가 다르고, 위치가 제각각입니다.")

## 8.2 학습 함수 — 여기만 판마다 다릅니다

In [ ]:
import keras
from keras import layers

def train(s, shape, kind, seed=42, epochs=12):
    """DNN과 CNN을 같은 조건에서. **이 함수만 판마다 다릅니다.**"""
    dlbook.set_seed(seed)
    if kind == "dnn":
        m = keras.Sequential([
            layers.Input(shape=shape), layers.Flatten(),
            layers.Dense(128, activation="relu"),
            layers.Dense(64, activation="relu"),
            layers.Dense(10, activation="softmax")])
    else:
        m = keras.Sequential([
            layers.Input(shape=shape),
            layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(2),
            layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(2),
            layers.Flatten(), layers.Dense(64, activation="relu"),
            layers.Dense(10, activation="softmax")])
    m.compile(optimizer=keras.optimizers.Adam(0.001),
              loss="sparse_categorical_crossentropy")
    m.fit(s.x_train, s.y_train, epochs=dlbook.smoke.epochs(epochs),
          batch_size=128, verbose=0)
    pred = m.predict(s.x_test, verbose=0, batch_size=256).argmax(1)
    return metrics.accuracy(s.y_test, pred), m.count_params()

## 8.3 나란히 놓고 봅니다

In [ ]:
print(f"{'데이터셋':<16}{'DNN':>9}{'CNN':>9}{'차이':>9}"
      f"{'DNN 파라미터':>14}{'CNN 파라미터':>14}")
print("-" * 71)
for name, fn, shape in SETS:
    s = fn()
    a_dnn, p_dnn = train(s, shape, "dnn")
    a_cnn, p_cnn = train(s, shape, "cnn")
    key = name.split("-")[0].lower() if name != "Fashion-MNIST" else "fashion"
    print(f"{name:<16}{a_dnn:>9.3f}{a_cnn:>9.3f}{a_cnn - a_dnn:>9.3f}"
          f"{p_dnn:>14,}{p_cnn:>14,}")
    dlbook.record(f"ch08d_{key}_dnn", a_dnn)
    dlbook.record(f"ch08d_{key}_cnn", a_cnn)
    dlbook.record(f"ch08d_{key}_gap", a_cnn - a_dnn)

print()
print("★ **어려워질수록 CNN의 이점이 커집니다.**")
print("  MNIST 0.01 → Fashion 0.03 → CIFAR-10 0.20")
print()
print("★ 그리고 CIFAR-10에서 CNN은 **파라미터가 절반도 안 되는데** 이깁니다.")
print("  8장 §8.4의 파라미터 공유입니다. 크기가 아니라 **구조**가 이겼습니다.")

## 정리

| 데이터셋 | DNN | CNN | 차이 |
|---|:--:|:--:|:--:|
| MNIST | 0.976 | 0.986 | 0.010 |
| Fashion-MNIST | 0.871 | 0.898 | 0.027 |
| **CIFAR-10** | **0.468** | **0.671** | **0.203** |

- **MNIST로는 CNN이 왜 필요한지가 안 드러납니다.** 차이가 0.01뿐입니다.
  숫자가 대체로 가운데 있어 `Flatten` 이 이웃 관계를 버려도 손해가
  적기 때문입니다.
- **CIFAR-10에서 격차가 20배로 벌어집니다.** 진짜 사진은 위치도
  각도도 제각각입니다. §8.3에서 도형을 ±6픽셀 흔들었을 때 본 것과
  같은 이야기입니다.
- **CIFAR-10의 CNN은 파라미터가 DNN의 절반도 안 됩니다**
  (167,562 대 402,250). 이기는 이유가 **크기가 아니라 구조**임을
  보여 줍니다.

### 연습

1. CIFAR-10에서 **CNN 층을 하나 더** 쌓으십시오. 얼마나 나아집니까.
2. 데이터 증강(4장 §4.5)을 켜면 어느 쪽이 더 이득을 봅니까.
3. CIFAR-10에서 0.67은 낮은 값입니다. **0.9를 넘기려면 무엇이
   필요합니까.** (9장을 미리 보셔도 좋습니다)
4. Fashion-MNIST의 차이(0.027)가 MNIST(0.010)보다 큰 이유를
   두 데이터셋의 그림을 그려 보고 설명하십시오.